# Comparing Marketing Campaigns with Expected Utility (SOLUTION)

## Scenario

You are the lead decision scientist at **BlueDoor Hosts**, a small
short-term-rental management firm that lists properties on Airbnb-style
platforms. The Head of Growth wants to launch a campaign next quarter to
attract new property owners onto the BlueDoor platform, and has put three
options on the table:

- **Premium Listing Push** — invest in professional photography, premium
  listing positioning, and paid placement on aggregator sites. Expensive,
  with high upside if the broader short-term-rental market is strong but a
  sizeable loss if the market softens.
- **Local Concierge Add-on** — bundle BlueDoor listings with a local
  concierge service (airport pickup, restaurant booking, late-night
  support). Modest budget, modest upside, modest downside.
- **Hold** — defer the campaign one quarter and reinvest the budget into
  existing-host retention.

BlueDoor operates across a set of mid-tier short-term-rental markets. The
campaign is a *single firm-wide choice* — whichever option the team picks
runs across BlueDoor's whole portfolio next quarter; we are not choosing
different campaigns for different cities. How well that single choice
performs depends on **the demand cycle the short-term-rental industry is
in over the next three months** — a Strong-demand, Average, or Weak-demand
environment that affects all of BlueDoor's markets at once. The cycle is
uncertain (we decide the campaign now; the cycle reveals itself later),
but it is not a coin flip — there is a real distribution we can estimate
from data.

To estimate that distribution, we use cross-city variation in current
Inside Airbnb data as a *stand-in* for the range of conditions any given
market can be in at any given time. The reasoning: eighteen real cities
span a wide range of revenue-proxy values today, and that cross-sectional
spread is an empirical picture of what a Strong, Average, or
Weak rental environment looks like.

The file `data/airbnb_city_stats.csv` contains city-level summary
statistics for 18 well-known short-term-rental markets in the format
published by Inside Airbnb (see `data/README.md` for the full citation).
For each city, we have the median nightly price, the median monthly
reviews per listing (Inside Airbnb's standard occupancy proxy), and a few
other fields. We compute `median_price × median_reviews_per_month` — a
per-listing monthly revenue proxy — for each city, tertile-bin the
eighteen cities into Strong / Average / Weak environments, and use the
empirical share of cities in each bin as the prior on what kind of
environment BlueDoor will face next quarter. The cities themselves are
not BlueDoor's markets; they are a benchmark for what "a Strong market"
or "a Weak market" looks like in numbers.

## What this notebook delivers

The Head of Growth has asked the team to run the numbers — expected value,
expected utility under risk aversion, and minimax regret — and to identify
which decision rule the team would lean on if forced to commit today. A
full stakeholder-facing recommendation will be assembled later in the
course, after additional analysis steps. Today's deliverable is:

1. The side-by-side comparison of what each decision rule selects.
2. A 1–2 sentence defended choice of decision rule.
3. An answer to *"How sensitive is the EV-max choice to our assumption that
   the next quarter looks like the cross-city baseline?"*

## Setup

In [1]:
import numpy as np
import pandas as pd

DATA_PATH = "../comparing-marketing-campaigns-starter/data/airbnb_city_stats.csv"

## 1. Load the data

In [2]:
cities = pd.read_csv(DATA_PATH)
display(cities.head())
cities.tail()

,city,country,total_listings,median_price_usd,median_reviews_per_month,pct_entire_home
0,New York City,United States,37000,200,0.70,52
1,Los Angeles,United States,30000,240,0.50,68
2,Austin,United States,13000,240,0.55,72
3,Vancouver,Canada,5500,170,0.55,55
4,Mexico City,Mexico,25000,60,0.85,80


,city,country,total_listings,median_price_usd,median_reviews_per_month,pct_entire_home
13,Madrid,Spain,22000,110,0.75,68
14,Rome,Italy,28000,125,0.85,75
15,Lisbon,Portugal,25000,100,1.00,75
16,Tokyo,Japan,12000,95,0.40,55
17,Sydney,Australia,15000,195,0.50,70


## 2. Per-listing monthly revenue proxy

Per-listing monthly revenue indicator: median nightly
price times Inside Airbnb's standard occupancy proxy (median monthly reviews
per active listing).

In [3]:
cities["revenue_proxy"] = (
    cities["median_price_usd"] * cities["median_reviews_per_month"]
)
cities[["city", "median_price_usd", "median_reviews_per_month",
        "revenue_proxy"]].sort_values("revenue_proxy").round(2)

,city,median_price_usd,median_reviews_per_month,revenue_proxy
5,Buenos Aires,55,0.65,35.75
16,Tokyo,95,0.40,38.00
10,Berlin,80,0.55,44.00
6,Rio de Janeiro,70,0.70,49.00
4,Mexico City,60,0.85,51.00
9,London,120,0.50,60.00
8,Paris,110,0.60,66.00
7,Cape Town,100,0.70,70.00
13,Madrid,110,0.75,82.50
3,Vancouver,170,0.55,93.50


## 3. Classify each city into Strong / Average / Weak by tertile

In [4]:
t33, t67 = cities["revenue_proxy"].quantile([1 / 3, 2 / 3])
f"Cutoffs — 33rd: {t33:.2f}, 67th: {t67:.2f}"


def classify(value: float) -> str:
    if value >= t67:
        return "Strong"
    if value >= t33:
        return "Average"
    return "Weak"


cities["environment"] = cities["revenue_proxy"].apply(classify)
cities[["city", "revenue_proxy", "environment"]]

,city,revenue_proxy,environment
0,New York City,140.00,Strong
1,Los Angeles,120.00,Strong
2,Austin,132.00,Strong
3,Vancouver,93.50,Average
4,Mexico City,51.00,Weak
5,Buenos Aires,35.75,Weak
6,Rio de Janeiro,49.00,Weak
7,Cape Town,70.00,Average
8,Paris,66.00,Average
9,London,60.00,Weak


## 4. Empirical probability of each environment

In [5]:
counts = cities["environment"].value_counts()
p = (counts / counts.sum()).reindex(["Strong", "Average", "Weak"])
p.round(3)

environment
Strong     0.333
Average    0.333
Weak       0.333
Name: count, dtype: float64

By construction, the tertile bins each contain ~1/3 of the cities. This
gives a simple baseline distribution to work from.

## 5. Payoff matrix

In [6]:
payoffs = pd.DataFrame(
    {
        "Strong":  {"Premium Listing Push": 10.0, "Local Concierge Add-on": 3.0, "Hold": 0.0},
        "Average": {"Premium Listing Push":  2.0, "Local Concierge Add-on": 1.5, "Hold": 0.0},
        "Weak":    {"Premium Listing Push": -6.0, "Local Concierge Add-on": 0.0, "Hold": 0.0},
    }
)
payoffs

,Strong,Average,Weak
Premium Listing Push,10.0,2.0,-6.0
Local Concierge Add-on,3.0,1.5,0.0
Hold,0.0,0.0,0.0


## 6. Expected value per option

In [7]:
ev = (payoffs * p).sum(axis=1)
ev.round(3)

Premium Listing Push      2.0
Local Concierge Add-on    1.5
Hold                      0.0
dtype: float64

## 7. Expected CRRA utility per option

CRRA utility on (wealth_baseline + incremental profit), with γ = 2 and
W = $50M. The wealth baseline keeps utility well-defined and represents
the existing scale of the business; with γ > 1 the function is more sensitive
to losses than to equally sized gains, which is what "risk aversion" means
operationally here.

In [8]:
GAMMA = 2.0
WEALTH_M = 50.0


def crra_utility(profit_m, gamma: float = GAMMA, wealth_m: float = WEALTH_M):
    """CRRA utility evaluated at (wealth_m + profit_m). Vectorized."""
    x = wealth_m + np.asarray(profit_m, dtype=float)
    if abs(gamma - 1.0) < 1e-9:
        return np.log(x)
    return (np.power(x, 1.0 - gamma) - 1.0) / (1.0 - gamma)


utility_table = payoffs.map(crra_utility)
expected_utility = (utility_table * p).sum(axis=1)
ce_total_wealth = np.power(
    expected_utility * (1.0 - GAMMA) + 1.0, 1.0 / (1.0 - GAMMA)
)
certainty_equivalent = ce_total_wealth - WEALTH_M

display(expected_utility.round(6))
certainty_equivalent.round(3)

Premium Listing Push      0.980458
Local Concierge Add-on    0.980572
Hold                      0.980000
dtype: float64

Premium Listing Push      1.173
Local Concierge Add-on    1.471
Hold                     -0.000
dtype: float64

Notice that *Premium Listing Push* has the highest **expected profit** but
*Local Concierge Add-on* has the highest **certainty-equivalent profit**.
A risk-averse decision-maker would trade away ~$0.5M of expected profit to
avoid the -$6M downside in a Weak environment.

## 8. Minimax regret per option

In [9]:
best_per_state = payoffs.max(axis=0)
regret = best_per_state - payoffs
max_regret = regret.max(axis=1)

display(regret)
max_regret

,Strong,Average,Weak
Premium Listing Push,0.0,0.0,6.0
Local Concierge Add-on,7.0,0.5,0.0
Hold,10.0,2.0,0.0


Premium Listing Push       6.0
Local Concierge Add-on     7.0
Hold                      10.0
dtype: float64

## 9. Compare the three decision rules

In [10]:
comparison = pd.DataFrame(
    {
        "Selected option": [ev.idxmax(),
                            expected_utility.idxmax(),
                            max_regret.idxmin()],
        "Score": [f"EV = ${ev.max():.2f}M",
                  f"CE = ${certainty_equivalent[expected_utility.idxmax()]:.2f}M",
                  f"Max regret = ${max_regret.min():.2f}M"],
    },
    index=["EV-max", "Expected-utility-max (γ=2)", "Minimax regret"],
)
comparison

,Selected option,Score
EV-max,Premium Listing Push,EV = $2.00M
Expected-utility-max (γ=2),Local Concierge Add-on,CE = $1.47M
Minimax regret,Premium Listing Push,Max regret = $6.00M


**The three rules do not agree.** EV-max picks Premium Listing Push,
minimax-regret picks Premium Listing Push, but expected CRRA utility (γ = 2)
picks Local Concierge Add-on. The disagreement is real: Premium has the
higher expected profit but a heavier tail in a Weak market — its
certainty-equivalent profit ($1.17M) is lower than Local Concierge's
($1.47M) once the −$6M downside is priced in.

**The decision rule I would lean on is expected CRRA utility.** With γ = 2,
the gap between Premium's EV ($2.0M) and Local Concierge's EV ($1.5M) does
not justify the −$6M downside in a Weak quarter for a firm of BlueDoor's
scale. A risk-neutral decision-maker should lean on EV-max instead.

This deliverable stops at *defending a decision rule*. Translating the
comparison into a stakeholder-facing recommendation — with caveats,
robustness checks, and downstream pipeline output — is built explicitly in
the communication-focused modules later in the course.

## 10. Sensitivity flex — elevated downside risk

In [11]:
p_alt = pd.Series({"Strong": 0.20, "Average": 0.30, "Weak": 0.50})
ev_alt = (payoffs * p_alt).sum(axis=1)
display(ev_alt.round(3))
ev.round(3)

Premium Listing Push     -0.40
Local Concierge Add-on    1.05
Hold                      0.00
dtype: float64

Premium Listing Push      2.0
Local Concierge Add-on    1.5
Hold                      0.0
dtype: float64

Under elevated downside-risk weighting, **Premium Listing Push flips to a
negative expected profit** and Local Concierge Add-on becomes the EV-max
option. The EV-max choice is sensitive to the assumed environment
probabilities — which is exactly the point of running the flex. The
headline takeaway is that any analytical output downstream of this
probability vector inherits its conditionality, and a stakeholder-facing
recommendation built on top of this exercise should explicitly state which
probability assumption it is conditional on.